# Session 2: distance = speed of light x travel time
**A. Real-world problem.** A GPS satellite cannot throw a tape measure to your phone. It sends
a radio signal stamped with the time it left. The phone notes the time it arrived. Radio
travels at the speed of light c, so distance = c x travel time. But c is huge, so a clock error
of one millionth of a second is a distance error of about 300 m. How do we turn such
distances into a position, and how good must the clocks be?

**Honest labels.** The satellite positions below are *toy* positions chosen for teaching.
They are not real orbits. The Earth is a sphere of radius 6371 km.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

C = 299_792_458.0          # speed of light in vacuum (m/s), exact by definition of the metre
R_EARTH = 6_371_000.0      # mean Earth radius (m)
R_ORBIT = 26_560_000.0     # distance of the toy satellites from the Earth's centre (m), close to GPS orbit size

## Step 1: time to distance, and back
A *pseudorange* is "c x (arrival time - send time)" as the receiver's clock sees it. It is
called *pseudo* because the clocks are not perfect, so it is not exactly the true distance.

In [ ]:
def range_from_time(t_seconds):
    """Distance (m) travelled by a radio signal in t seconds."""
    return C * t_seconds


def time_error_to_range_error(dt_seconds):
    """Range error (m) caused by a timing error dt (s)."""
    return C * dt_seconds


def clock_accuracy_for(range_error_m):
    """Largest timing error (s) that keeps the range error within range_error_m."""
    return range_error_m / C

## Step 2: subtract the equations to make them linear
Each satellite i at position s_i gives one equation |x - s_i| = r_i, which is quadratic in x.
Subtracting the equation of satellite 1 from satellite i cancels the |x|^2 term:

    2 (s_i - s_1) . x = |s_i|^2 - |s_1|^2 - r_i^2 + r_1^2

That is a straight line (2D) or plane (3D). Stack these as A x = b and solve.

In [ ]:
def linear_system(sats, ranges):
    """Build A and b of the subtracted equations (rows = satellites 2..n)."""
    sats, ranges = np.asarray(sats, float), np.asarray(ranges, float)
    A = 2.0 * (sats[1:] - sats[0])
    b = (sats[1:] ** 2).sum(axis=1) - (sats[0] ** 2).sum() - ranges[1:] ** 2 + ranges[0] ** 2
    return A, b


def linear_fix(sats, ranges):
    """Position from exact-or-noisy ranges by the subtraction trick (least squares if over-determined)."""
    A, b = linear_system(sats, ranges)
    x, *_ = np.linalg.lstsq(A, b, rcond=None)
    return x

## Step 3: toy satellites in 3D, and why three are not enough
`toy_sats` places satellites at chosen directions (azimuth, elevation) as seen by a receiver on
the Earth's surface, then puts each one at distance R_ORBIT from the Earth's centre.

In [ ]:
def receiver_on_earth(lat_deg, lon_deg):
    la, lo = np.radians(lat_deg), np.radians(lon_deg)
    return R_EARTH * np.array([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)])


def local_axes(rec):
    """East, north and up unit vectors at the receiver."""
    up = rec / np.linalg.norm(rec)
    east = np.cross([0.0, 0.0, 1.0], up)
    east /= np.linalg.norm(east)
    return east, np.cross(up, east), up


def toy_sats(rec, az_el_deg, radius=R_ORBIT):
    """Toy satellite positions (m) in directions (azimuth, elevation) in degrees from the receiver."""
    e, n, u = local_axes(rec)
    out = []
    for az, el in az_el_deg:
        a, h = np.radians(az), np.radians(el)
        d = np.cos(h) * np.sin(a) * e + np.cos(h) * np.cos(a) * n + np.sin(h) * u
        # solve |rec + t d| = radius for the positive t
        bq = rec @ d
        t = -bq + np.sqrt(bq * bq - rec @ rec + radius * radius)
        out.append(rec + t * d)
    return np.array(out)


def three_sphere_points(sats3, ranges3):
    """The (up to) two points at the given distances from three satellites."""
    A, b = linear_system(sats3, ranges3)            # 2 equations, 3 unknowns
    x0, *_ = np.linalg.lstsq(A, b, rcond=None)      # one solution of the two planes
    n = np.cross(A[0], A[1])
    n /= np.linalg.norm(n)                          # direction along the line where the planes meet
    w = x0 - sats3[0]
    bq = w @ n
    disc = bq * bq - (w @ w - ranges3[0] ** 2)
    if disc < 0:
        return np.empty((0, 3))
    s = np.sqrt(disc)
    return np.array([x0 + (-bq + s) * n, x0 + (-bq - s) * n])

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("1 microsecond of timing error =", round(time_error_to_range_error(1e-6), 2), "m of range error")
    print("clock accuracy for a 10 m range error:", round(clock_accuracy_for(10.0) * 1e9, 2), "ns")
    print("travel time from a satellite straight overhead:", round((R_ORBIT - R_EARTH) / C * 1e3, 2), "ms")

    rec = receiver_on_earth(19.0, 73.0)
    azel = [(20, 60), (110, 35), (200, 50), (290, 25)]
    sats = toy_sats(rec, azel)
    r = np.linalg.norm(sats - rec, axis=1)
    print("toy ranges (km):", np.round(r / 1e3, 1))
    fix = linear_fix(sats, r)
    print("4 exact ranges -> position error (m):", float(np.linalg.norm(fix - rec)))
    pts = three_sphere_points(sats[:3], r[:3])
    for p in pts:
        print("3 satellites: a solution at distance from Earth's centre =", round(np.linalg.norm(p) / 1e3, 1), "km; error",
              round(float(np.linalg.norm(p - rec)) / 1e3, 1), "km")

    r_bad = r.copy()
    r_bad[0] += time_error_to_range_error(1e-6)
    print("1 us error on one range -> position error (m):", round(float(np.linalg.norm(linear_fix(sats, r_bad) - rec)), 1))
    r_off = r + time_error_to_range_error(1e-3)
    print("receiver clock 1 ms fast on all ranges -> position error (km):", round(float(np.linalg.norm(linear_fix(sats, r_off) - rec)) / 1e3, 1))

    dts = np.linspace(0, 1e-6, 21)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(dts * 1e6, time_error_to_range_error(dts), "o-", ms=3, label="range error = c x dt")
    errs = []
    for dt in dts:
        rr = r.copy()
        rr[0] += time_error_to_range_error(dt)
        errs.append(np.linalg.norm(linear_fix(sats, rr) - rec))
    ax.plot(dts * 1e6, errs, "s-", ms=3, label="position error (toy geometry)")
    ax.set_xlabel("timing error on one satellite (microseconds)")
    ax.set_ylabel("error (m)")
    ax.set_title("1 microsecond is about 300 m of range")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "timing_error.png", dpi=120)

    # two points where three spheres meet: one on the Earth, one far away
    fig2 = plt.figure(figsize=(6, 5))
    ax3 = fig2.add_subplot(projection="3d")
    u, v = np.meshgrid(np.linspace(0, 2 * np.pi, 30), np.linspace(0, np.pi, 16))
    ax3.plot_wireframe(R_EARTH * np.cos(u) * np.sin(v) / 1e6, R_EARTH * np.sin(u) * np.sin(v) / 1e6,
                       R_EARTH * np.cos(v) / 1e6, color="0.8", lw=0.4)
    ax3.scatter(*(sats[:3].T / 1e6), c="k", marker="s", label="3 toy satellites")
    ax3.scatter(*(pts.T / 1e6), c="r", s=40, label="the two solutions")
    ax3.set_xlabel("x (1000 km)")
    ax3.set_ylabel("y (1000 km)")
    ax3.set_zlabel("z (1000 km)")
    ax3.set_title("Three spheres meet at two points")
    ax3.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "three_spheres.png", dpi=120)

## Where this stops
We assumed the receiver's clock is perfect. A phone clock is a cheap quartz crystal and is off by
far more than 1 microsecond. Session 3 treats the clock error as one more unknown.

## Coding tasks
1. **Easy.** Print the range error for 1 ms, 1 us and 1 ns of timing error and say each in words (km, m, cm).
2. **Stretch.** Change the elevations in the 3D example so all four toy satellites are low (about 20 degrees). Print the position error for a 1 microsecond error on one range and compare with 642 m.
3. **Explain.** Explain why the subtracted equation is a plane (a line in 2D), and what two planes do when they meet.

## Assessment
- **A. Conceptual.** Why does subtracting one sphere equation from another make the problem linear, and what do the new equations represent geometrically?
- **B. Numerical.** A satellite clock is wrong by 50 nanoseconds. What range error is that? Is it small compared with a 10 m accuracy goal?
- **C. Graph and simulation interpretation.** In the timing-error figure the position-error line is steeper than the range-error line. What does the ratio of slopes tell you, and what could make it larger?
- **D. Design and optimisation challenge.** Design: you need the position error below 10 m, and in the toy geometry of the figure the amplification is about 2.14. How accurate (in nanoseconds) must the clock error on one range be? What steadiness over one day (a fraction) does that need?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("1 microsecond is 299.79 m of range", abs(L.time_error_to_range_error(1e-6) - 299.792458) < 1e-6))
    checks.append(("range_from_time and clock_accuracy_for are inverses", abs(L.clock_accuracy_for(L.range_from_time(2.5e-3)) - 2.5e-3) < 1e-15))
    dt10 = L.clock_accuracy_for(10.0)
    checks.append((f"design: 10 m needs a clock better than 33.4 ns (got {dt10 * 1e9:.2f} ns)", abs(dt10 * 1e9 - 33.36) < 0.01))
    stab = dt10 / 86400.0
    checks.append((f"design: holding 33.4 ns for a day is 3.9e-13 steadiness (got {stab:.2e})", abs(stab - 3.86e-13) < 0.02e-13))

    # worked example: three ground beacons (km -> m), travel times in microseconds, linear solve
    beacons = np.array([[0.0, 40.0], [120.0, 40.0], [60.0, 95.0]]) * 1e3
    me = np.array([45.0, 25.0]) * 1e3
    t = np.linalg.norm(beacons - me, axis=1) / L.C
    checks.append(("worked example: travel times are 158.2, 255.1 and 238.8 microseconds",
                   np.allclose(t * 1e6, [158.22, 255.12, 238.80], atol=0.01)))
    ranges = L.range_from_time(t)
    A, b = L.linear_system(beacons, ranges)
    checks.append(("worked example: first subtracted equation is 240 x = 10800 (km units)",
                   abs(A[0, 0] / 1e3 - 240.0) < 1e-6 and abs(A[0, 1]) < 1e-9 and abs(b[0] / 1e6 - 10800.0) < 1e-3))
    fix = L.linear_fix(beacons, ranges)
    checks.append(("worked example: the linear solve gives (45, 25) km", np.allclose(fix, me, atol=1e-3)))

    # 3D with toy satellites
    rec = L.receiver_on_earth(19.0, 73.0)
    sats = L.toy_sats(rec, [(20, 60), (110, 35), (200, 50), (290, 25)])
    checks.append(("toy satellites are at the orbit radius", np.allclose(np.linalg.norm(sats, axis=1), L.R_ORBIT)))
    r = np.linalg.norm(sats - rec, axis=1)
    checks.append(("four exact ranges recover the 3D position to 1 mm", np.linalg.norm(L.linear_fix(sats, r) - rec) < 1e-3))
    pts = L.three_sphere_points(sats[:3], r[:3])
    radii = sorted(np.linalg.norm(pts, axis=1) / 1e3)
    checks.append(("three spheres give two points: one on the Earth, one far out in space",
                   len(pts) == 2 and abs(radii[0] - 6371.0) < 0.01 and radii[1] > 30000.0))
    # a 1 microsecond error on one range moves the fix by hundreds of metres
    rb = r.copy()
    rb[0] += L.time_error_to_range_error(1e-6)
    e = np.linalg.norm(L.linear_fix(sats, rb) - rec)
    checks.append((f"1 microsecond on one range moves the fix by more than 300 m (got {e:.0f} m)", e > 299.0))
    # a receiver clock error on all ranges is large and not zero
    e2 = np.linalg.norm(L.linear_fix(sats, r + L.range_from_time(1e-3)) - rec)
    checks.append(("a 1 ms receiver clock error ruins the fix (tens of km)", e2 > 10e3))
    # edge case: collinear beacons cannot fix a 2D position
    line = np.array([[0.0, 0.0], [1.0, 0.0], [2.0, 0.0]])
    A2, _ = L.linear_system(line, np.array([1.0, 1.0, 1.0]))
    checks.append(("collinear beacons give a rank-deficient system", np.linalg.matrix_rank(A2) < 2))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
